# Tải 3 file Fliggy Transfer-Plans qua Google Colab

Nguồn: https://tianchi.aliyun.com/dataset/140721

| File | Dung lượng hiển thị trên Tianchi (tham khảo) |
|---|---:|
| `user_behavior_transfer_plans.csv` | 1.38 GB |
| `user_profile.csv` | 43.80 MB |
| `transfer_plan_profile.csv` | 677.44 MB |

**Cách dùng:** upload notebook này lên Google Colab, kết nối runtime CPU và chạy từng ô.
Lấy **Copy download link** riêng cho từng file sau khi đăng nhập Tianchi, rồi dán vào ba ô nhập ẩn.
Chấp nhận URL thuần, `<URL>` hoặc Markdown `[nhãn](URL)`.

Notebook tự đọc kích thước, ETag và CRC64 từ OSS, tải lần lượt ba file, thử lại và tải tiếp khi gián đoạn.
File được đánh dấu hoàn tất sau khi kiểm tra dung lượng và checksum máy chủ (nếu có).
Phần tải dở chỉ tiếp tục được khi file `.part` và metadata vẫn còn.

Có thể bật lưu vào Google Drive ở ô cấu hình để giữ phần tải dở qua các phiên Colab.
Link có chữ ký chỉ được nhập trong runtime và dùng để truy cập OSS; không ghi vào notebook hay metadata.
Tốc độ từ Colab cần đo thực tế, không mặc định nhanh hơn mạng của bạn.

In [ ]:
# Chỉ cài thư viện còn thiếu trong runtime hiện tại.
import importlib.util
import subprocess
import sys

packages = ["requests", "tqdm", "crcmod", "ipywidgets"]
missing = [name for name in packages if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
from pathlib import Path
from getpass import getpass
from urllib.parse import urlsplit, parse_qs, unquote
from itertools import islice
import csv
import html
import json
import re
import shutil
import time

import crcmod
import requests
from tqdm.auto import tqdm
from IPython.display import Markdown, display

DATASET_ID = "140721"
FILENAMES = (
    "user_behavior_transfer_plans.csv",
    "user_profile.csv",
    "transfer_plan_profile.csv",
)
SAVE_TO_DRIVE = False  # Đổi thành True trước khi chạy ô này nếu muốn lưu vào Drive.

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/fliggy/140721")
elif Path("/content").is_dir():
    OUTPUT_DIR = Path("/content/fliggy/140721")
else:
    OUTPUT_DIR = Path.cwd() / "data" / "fliggy" / "140721"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Thư mục lưu dữ liệu:", OUTPUT_DIR)

## 1. Nhận link của đúng 3 file

Dùng link từ **dataset 140721**. `user_profile.csv` ở dataset 113649 là file khác.
Nếu link hết hạn, lấy link mới và chạy lại ô nhập; nhấn Enter để giữ link cũ của các file khác.
Nếu tải một file lỗi, notebook vẫn thử tải các file còn lại.

In [ ]:
class DownloadError(RuntimeError):
    pass


def normalize_download_url(raw, filename):
    text = html.unescape(raw.strip())
    for character in "\u200b\u200c\u200d\u2060\ufeff":
        text = text.replace(character, "")
    markdown = re.search(r"\]\(\s*(https?://[^\s<>]+?)\s*\)", text)
    plain = re.search(r'https?://[^\s<>\[\]()"`·•]+', text)
    if markdown:
        candidate = markdown.group(1)
    elif plain:
        candidate = plain.group(0)
    else:
        raise ValueError("Không tìm thấy URL. Hãy dán Copy download link đầy đủ.")
    candidate = re.sub(r"\\([&\[\]()<>])", r"\1", candidate)
    parsed = urlsplit(candidate)
    host = parsed.hostname or ""
    if (parsed.scheme != "https" or not host.startswith("tianchi-")
            or not host.endswith(".aliyuncs.com") or parsed.username or parsed.password):
        raise ValueError("Cần link HTTPS trực tiếp từ bucket Tianchi trên OSS.")
    expected_path = f"/file/opensearch/documents/{DATASET_ID}/{filename}"
    if unquote(parsed.path) != expected_path:
        raise ValueError(f"Cần link của {filename} thuộc dataset {DATASET_ID}.")
    expires = parse_qs(parsed.query).get("Expires", [])
    if expires:
        try:
            expired = int(expires[0]) <= time.time()
        except ValueError:
            raise ValueError("Tham số Expires không hợp lệ. Sao chép lại đầy đủ link.") from None
        if expired:
            raise ValueError("Link đã hết hạn. Lấy Copy download link mới từ Tianchi.")
    return candidate

In [ ]:
LINKS = globals().get("LINKS", {})
for filename in FILENAMES:
    while True:
        raw = getpass(f"{filename} — dán link (Enter để giữ link cũ): ")
        if not raw.strip():
            raw = LINKS.get(filename, "")
        try:
            LINKS[filename] = normalize_download_url(raw, filename)
            print(f"Đã nhận link: {filename}")
            break
        except ValueError as error:
            print(error)

## 2. Đọc thông tin chính xác từ máy chủ

Dùng GET Range 1 byte để đọc thông tin từ link có chữ ký.
Không dùng các dung lượng đã làm tròn trong ảnh làm điều kiện kiểm tra.

In [ ]:
def check_http(response):
    status = response.status_code
    if status in (401, 403):
        raise DownloadError("Link hết hạn hoặc không còn quyền tải. Nhập lại link Tianchi mới.")
    if status == 404:
        raise DownloadError("Máy chủ không tìm thấy file.")
    if status == 412:
        raise DownloadError("File nguồn đã thay đổi ETag; không nối vào phần tải cũ.")
    if status >= 400:
        raise requests.HTTPError(f"HTTP {status}")


def read_metadata(url, filename, attempts=3):
    for attempt in range(attempts):
        try:
            with requests.get(url, headers={"Range": "bytes=0-0", "Accept-Encoding": "identity"},
                              stream=True, timeout=(15, 30)) as response:
                check_http(response)
                if response.status_code == 206:
                    match = re.fullmatch(r"bytes 0-0/(\d+)", response.headers.get("Content-Range", ""))
                    if not match:
                        raise DownloadError("Content-Range của yêu cầu metadata không hợp lệ.")
                    size = int(match.group(1))
                elif response.status_code == 200:
                    size = int(response.headers.get("Content-Length", "0"))
                else:
                    raise DownloadError("Không đọc được thông tin file từ OSS.")
                etag = response.headers.get("ETag")
                crc = response.headers.get("x-oss-hash-crc64ecma")
                if size <= 0 or not etag:
                    raise DownloadError("Thiếu kích thước hoặc ETag; chưa thể tải tiếp an toàn.")
                if response.headers.get("Content-Encoding", "identity").lower() != "identity":
                    raise DownloadError("Máy chủ trả dữ liệu nén HTTP không phù hợp với tải theo byte.")
                return {"dataset_id": DATASET_ID, "filename": filename, "size": size,
                        "etag": etag, "crc64": int(crc) if crc is not None else None}
        except requests.RequestException as error:
            print(f"{filename}: lỗi kết nối metadata ({type(error).__name__}), lần {attempt+1}/{attempts}.")
            if attempt + 1 < attempts:
                time.sleep(2)
    raise DownloadError("Chưa đọc được metadata. Chạy lại ô metadata khi kết nối ổn định.")


METADATA = {}
for filename in FILENAMES:
    try:
        METADATA[filename] = read_metadata(LINKS[filename], filename)
        meta = METADATA[filename]
        print(f"{filename}: {meta['size']:,} byte ({meta['size']/1024**2:,.2f} MiB); CRC64: {'có' if meta['crc64'] is not None else 'không có'}")
    except (DownloadError, ValueError) as error:
        print(f"{filename}: {error}")
print(f"Đọc được metadata {len(METADATA)}/{len(FILENAMES)} file.")

## 3. Hàm tải tiếp và kiểm tra dữ liệu

Mỗi file có `.part` và `.meta.json` riêng. Metadata không chứa URL hay token.
Nếu máy chủ thay đổi file, notebook dừng file đó để tránh nối dữ liệu khác phiên bản.
CRC64 được tính bằng thư viện `crcmod`, có thanh tiến độ riêng.

In [ ]:
def verify_file(path, meta):
    if path.stat().st_size != meta["size"]:
        raise DownloadError("Dung lượng file chưa khớp metadata máy chủ.")
    if meta["crc64"] is None:
        return False
    checksum = crcmod.Crc(0x142F0E1EBA9EA3693, initCrc=0, rev=True, xorOut=0xFFFFFFFFFFFFFFFF)
    with path.open("rb") as stream, tqdm(total=meta["size"], unit="B", unit_scale=True,
                                       desc=f"CRC64 {meta['filename']}") as progress:
        while block := stream.read(4 * 1024 * 1024):
            checksum.update(block)
            progress.update(len(block))
    if checksum.crcValue != meta["crc64"]:
        raise DownloadError("CRC64 không khớp. Giữ file để kiểm tra; đổi tên phần .part trước khi tải mới.")
    return True


def download_one(url, destination, meta, attempts=8):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".part")
    metadata_path = destination.with_name(destination.name + ".meta.json")
    if metadata_path.exists():
        previous = json.loads(metadata_path.read_text(encoding="utf-8"))
        if previous != meta and (partial.exists() or destination.exists()):
            raise DownloadError("Metadata nguồn đã đổi. Giữ file cũ và chọn thư mục tải khác.")
    elif partial.exists():
        raise DownloadError("Có file .part nhưng thiếu .meta.json; không nối vào dữ liệu chưa xác định nguồn.")

    if destination.exists():
        crc_ok = verify_file(destination, meta)
        metadata_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
        return {"path": str(destination), "size": meta["size"], "crc64_verified": crc_ok}
    metadata_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    if partial.exists() and partial.stat().st_size > meta["size"]:
        raise DownloadError("File .part lớn hơn nguồn; cần kiểm tra file tải dở.")
    remaining = meta["size"] - (partial.stat().st_size if partial.exists() else 0)
    if shutil.disk_usage(destination.parent).free < remaining + 100 * 1024**2:
        raise DownloadError("Không đủ dung lượng trống để tải file này.")

    started = time.monotonic()
    initial = partial.stat().st_size if partial.exists() else 0
    with requests.Session() as session:
        for attempt in range(attempts):
            offset = partial.stat().st_size if partial.exists() else 0
            if offset == meta["size"]:
                crc_ok = verify_file(partial, meta)
                partial.rename(destination)
                return {"path": str(destination), "size": meta["size"], "crc64_verified": crc_ok}
            headers = {"Accept-Encoding": "identity", "If-Match": meta["etag"]}
            if offset:
                headers["Range"] = f"bytes={offset}-"
            try:
                with session.get(url, headers=headers, stream=True, timeout=(20, 30)) as response:
                    check_http(response)
                    if response.headers.get("ETag") != meta["etag"]:
                        raise DownloadError("ETag không khớp metadata. Chạy lại bước đọc metadata để kiểm tra.")
                    if response.headers.get("Content-Encoding", "identity").lower() != "identity":
                        raise DownloadError("Dữ liệu nén HTTP không phù hợp với tải tiếp theo byte.")
                    if offset or response.status_code == 206:
                        match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", response.headers.get("Content-Range", ""))
                        if response.status_code != 206 or not match:
                            raise DownloadError("Máy chủ không hỗ trợ tải tiếp cho yêu cầu này; giữ phần đã tải.")
                        if tuple(map(int, match.groups())) != (offset, meta["size"]-1, meta["size"]):
                            raise DownloadError("Vị trí Range không khớp file tải dở.")
                    elif response.status_code != 200:
                        raise DownloadError("Trạng thái HTTP tải file không hợp lệ.")
                    if int(response.headers.get("Content-Length", "-1")) != meta["size"] - offset:
                        raise DownloadError("Dung lượng trả về không khớp phần còn thiếu.")
                    with partial.open("ab") as stream, tqdm(total=meta["size"], initial=offset,
                            unit="B", unit_scale=True, desc=meta["filename"]) as progress:
                        for chunk in response.iter_content(chunk_size=64 * 1024):
                            if chunk:
                                if stream.tell() + len(chunk) > meta["size"]:
                                    raise DownloadError("Số byte trả về vượt kích thước dự kiến.")
                                stream.write(chunk)
                                progress.update(len(chunk))
                if partial.stat().st_size == meta["size"]:
                    crc_ok = verify_file(partial, meta)
                    partial.rename(destination)
                    elapsed = max(time.monotonic()-started, 0.001)
                    print(f"Hoàn tất {meta['filename']}; tốc độ trung bình lượt này: {(meta['size']-initial)/elapsed/1024:.1f} KiB/s")
                    return {"path": str(destination), "size": meta["size"], "crc64_verified": crc_ok}
            except requests.RequestException as error:
                print(f"{meta['filename']}: kết nối gián đoạn ({type(error).__name__}), lần {attempt+1}/{attempts}; giữ .part.")
            if attempt + 1 < attempts:
                time.sleep(min(2 ** attempt, 8))
    raise DownloadError("Chưa tải xong. Chạy lại ô tải để tiếp tục từ .part.")

## 4. Tải lần lượt cả 3 file

Mỗi file có tiến độ riêng. Có thể dừng và chạy lại ô này; file hoàn tất được kiểm tra lại,
file dở được tải tiếp. Nếu link hết hạn, chạy lại ô nhập link và metadata trước khi chạy lại ô tải.

In [ ]:
VERIFIED_FILES = {}
DOWNLOAD_ERRORS = {}
for filename in FILENAMES:
    if filename not in METADATA:
        DOWNLOAD_ERRORS[filename] = "Chưa đọc được metadata; kiểm tra link và chạy lại bước 2."
        print(f"Bỏ qua {filename}: chưa có metadata.")
        continue
    try:
        result = download_one(LINKS[filename], OUTPUT_DIR / filename, METADATA[filename])
        VERIFIED_FILES[filename] = result
    except (DownloadError, OSError, ValueError) as error:
        DOWNLOAD_ERRORS[filename] = str(error)
        print(f"Chưa hoàn tất {filename}: {error}")
print(f"Hoàn tất {len(VERIFIED_FILES)}/{len(FILENAMES)} file.")

## 5. Kiểm tra kết quả

Hiển thị trạng thái từng file và số trường của 5 dòng đầu, không nạp toàn bộ CSV vào RAM.
Nếu máy chủ không cung cấp CRC64, trạng thái sẽ ghi rõ chỉ kiểm tra dung lượng/ETag.

In [ ]:
for filename in FILENAMES:
    result = VERIFIED_FILES.get(filename)
    if result is None:
        print(f"{filename}: CHƯA XONG — {DOWNLOAD_ERRORS.get(filename, 'Chưa chạy ô tải.')}")
        continue
    state = "kích thước + CRC64 khớp" if result["crc64_verified"] else "đủ byte/ETag; máy chủ không có CRC64"
    with Path(result["path"]).open(encoding="utf-8-sig", newline="") as stream:
        rows = list(islice(csv.reader(stream), 5))
    print(f"{filename}: {result['size']:,} byte; {state}; số trường 5 dòng đầu: {[len(row) for row in rows]}")

## 6. Lấy dữ liệu về máy

- Nếu bật `SAVE_TO_DRIVE`, file đã ở `MyDrive/fliggy/140721`; có thể tải từ Google Drive về máy.
- Nếu lưu trong runtime, chọn từng file đã hoàn tất ở danh sách dưới và bấm **Tải về máy**.
  Lần lượt hoàn tất mỗi lượt tải trước khi chọn file tiếp theo.
- Trên máy, có thể lưu cả 3 CSV vào `data/fliggy/140721/`.

In [ ]:
import ipywidgets as widgets

if not VERIFIED_FILES:
    print("Chưa có file hoàn tất để tải về máy.")
else:
    selector = widgets.Dropdown(options=list(VERIFIED_FILES), description="File:", layout=widgets.Layout(width="650px"))
    button = widgets.Button(description="Tải về máy", button_style="success")
    status = widgets.Output()
    def export_file(_):
        with status:
            status.clear_output()
            filename = selector.value
            path = Path(VERIFIED_FILES[filename]["path"])
            if not path.is_file() or path.stat().st_size != VERIFIED_FILES[filename]["size"]:
                print("File đã thay đổi hoặc không còn trong runtime; chạy lại ô tải/kiểm tra.")
                return
            try:
                from google.colab import files
            except ImportError:
                print("File nằm tại:", path.resolve())
                return
            print("Bắt đầu tải về máy:", filename)
            files.download(str(path))
    button.on_click(export_file)
    display(selector, button, status)